## Import Datas

In [ ]:
import os
from google.colab import drive
drive.mount('/content/drive')
folder = "/content/drive/MyDrive/ICATH 2026/code/data/preprocessed options datas/"

Mounted at /content/drive


In [2]:
import pandas as pd
pd.set_option('display.float_format', '{:.4f}'.format)
pd.set_option('display.max_columns', None)
import numpy as np
import random
import torch

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

In [ ]:
TechnologyUnderlying = [ "AAPL", "AMZN", "GOOGL", "META", "MSFT" ]
AutomobileUnderlying = [ "F", "GM", "LCID", "RIVN", "TSLA" ]

listTickers = AutomobileUnderlying + TechnologyUnderlying

In [ ]:
dataset = {
    "AAPL" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "AMZN" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "GOOGL" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "META" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "MSFT" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "F" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "GM" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "LCID" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "RIVN" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
    "TSLA" : {"train" :pd.DataFrame(), "test" : pd.DataFrame()},
}

In [5]:
for ticker in dataset.keys():
    train_path = os.path.join(folder, f"{ticker}_train.csv")
    test_path = os.path.join(folder, f"{ticker}_test.csv")

    if not os.path.exists(train_path):
        print(f"Missing: {train_path}")
        continue

    train_dataset = pd.read_csv(train_path)
    test_dataset = pd.read_csv(test_path)

    train_dataset = train_dataset.fillna(train_dataset.mean(numeric_only=True))
    test_dataset = test_dataset.fillna(test_dataset.mean(numeric_only=True))

    dataset[ticker]["train"] = train_dataset
    dataset[ticker]["test"] = test_dataset

## Define functions

In [6]:
!pip install optuna

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 419.5/419.5 kB 22.7 MB/s eta 0:00:00


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import numpy as np
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import cross_val_score
import optuna

In [ ]:
test_size=0.2
random_state=42
target = 'lastPrice'
list_histos_datas_inputs = ["strike", "impliedVolatility", "timeToMaturity",
                            "riskFreeRate", "underlyingLastPrice"]

feature_combinations = [
    [],

    ["GS_" + ticker], ["SC_" + ticker], ["VIX"], ["PCR"],

    ["GS_" + ticker , "SC_" + ticker], ["GS_" + ticker , "VIX"], ["GS_" + ticker , "PCR"], ["SC_" + ticker , "VIX"], ["SC_" + ticker , "PCR"], ["VIX" , "PCR"],

    ["GS_" + ticker , "SC_" + ticker , "VIX"], ["GS_" + ticker , "SC_" + ticker , "PCR"], ["GS_" + ticker , "VIX" , "PCR"], ["SC_" + ticker , "VIX" , "PCR"],

	["GS_" + ticker , "SC_" + ticker , "VIX" , "PCR"]
]

In [ ]:
def train_rf_with_optuna(X_train, y_train, X_test, y_test, best_params):
    """
    Train a RandomForest using Optuna best_params.
    """

    model = RandomForestRegressor(
        n_estimators=best_params["n_estimators"],
        max_depth=best_params["max_depth"],
        max_features=best_params["max_features"],
        min_samples_split=best_params["min_samples_split"],
        min_samples_leaf=best_params["min_samples_leaf"],
        random_state=42,
        n_jobs=-1
    )

    model.fit(X_train, y_train.ravel())

    y_pred = model.predict(X_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    nrmse = rmse / np.mean(y_test)

    return mae, rmse, nrmse


def objective_for_rf(trial, X_train, y_train):
    n_estimators = trial.suggest_int("n_estimators", 100, 800)
    max_depth = trial.suggest_int("max_depth", 5, 50)
    min_samples_split = trial.suggest_int("min_samples_split", 2, 20)
    min_samples_leaf = trial.suggest_int("min_samples_leaf", 1, 10)
    max_features = trial.suggest_categorical("max_features", ["sqrt", "log2"])
    bootstrap = trial.suggest_categorical("bootstrap", [True, False])

    model = RandomForestRegressor(
        n_estimators=n_estimators,
        max_depth=max_depth,
        min_samples_split=min_samples_split,
        min_samples_leaf=min_samples_leaf,
        max_features=max_features,
        bootstrap=bootstrap,
        n_jobs=1
    )

    score = cross_val_score(model, X_train, y_train, cv=5, scoring="neg_mean_squared_error")
    return -score.mean()

def predict_current_price_using_rf(option_type, ticker, list_histos_datas_inputs=list_histos_datas_inputs, feature_combinations=feature_combinations):
  for proxy in feature_combinations:
    train_dataset = dataset[ticker]["train"]
    train_dataset = train_dataset[train_dataset["optionType"] == option_type]
    X_train = train_dataset[list_histos_datas_inputs + proxy].values
    y_train = train_dataset[["lastPrice"]].values

    test_dataset = dataset[ticker]["test"]
    test_dataset = test_dataset[test_dataset["optionType"] == option_type]
    X_test = test_dataset[list_histos_datas_inputs+ proxy].values
    y_test = test_dataset[["lastPrice"]].values

    #Normaliser les dataset
    scaler = StandardScaler()
    X_train = scaler.fit_transform(X_train)
    X_test = scaler.fit_transform(X_test)

    study = study = optuna.create_study(direction="minimize", sampler=optuna.samplers.TPESampler(seed=SEED))
    study.optimize(lambda trial: objective_for_rf(trial, X_train, y_train), n_trials=10)

    best_params = study.best_params
    #print(best_params)

    mae, rmse, nrmse = train_rf_with_optuna(
        X_train, y_train,
        X_test, y_test,
        best_params
    )
    print(f"{proxy} for {ticker} => (MAE={mae:.3f}; RMSE={rmse:.3f}; ; NRMSE={nrmse:.3f})")


## option using random forest

In [10]:
ticker = "TSLA"

In [18]:
predict_current_price_using_rf("put", ticker)

['PCR'] for TSLA => (MAE=1.813; RMSE=4.343; ; NRMSE=0.099)
['GS_TSLA', 'SC_TSLA'] for TSLA => (MAE=1.969; RMSE=5.173; ; NRMSE=0.118)
['GS_TSLA', 'VIX'] for TSLA => (MAE=1.919; RMSE=4.988; ; NRMSE=0.114)
['GS_TSLA', 'PCR'] for TSLA => (MAE=1.974; RMSE=4.893; ; NRMSE=0.112)
['SC_TSLA', 'VIX'] for TSLA => (MAE=1.921; RMSE=4.980; ; NRMSE=0.114)
['SC_TSLA', 'PCR'] for TSLA => (MAE=2.042; RMSE=5.123; ; NRMSE=0.117)
['VIX', 'PCR'] for TSLA => (MAE=1.990; RMSE=5.071; ; NRMSE=0.116)
['GS_TSLA', 'SC_TSLA', 'VIX'] for TSLA => (MAE=1.925; RMSE=4.857; ; NRMSE=0.111)
['GS_TSLA', 'SC_TSLA', 'PCR'] for TSLA => (MAE=2.037; RMSE=5.046; ; NRMSE=0.115)
['GS_TSLA', 'VIX', 'PCR'] for TSLA => (MAE=1.950; RMSE=4.816; ; NRMSE=0.110)
['SC_TSLA', 'VIX', 'PCR'] for TSLA => (MAE=1.955; RMSE=4.775; ; NRMSE=0.109)
['GS_TSLA', 'SC_TSLA', 'VIX', 'PCR'] for TSLA => (MAE=1.726; RMSE=4.215; ; NRMSE=0.096)


In [12]:
predict_current_price_using_rf("call", ticker)

['PCR'] for TSLA => (MAE=2.378; RMSE=4.041; ; NRMSE=0.051)
['GS_TSLA', 'SC_TSLA'] for TSLA => (MAE=2.703; RMSE=4.449; ; NRMSE=0.056)
['GS_TSLA', 'VIX'] for TSLA => (MAE=2.719; RMSE=4.453; ; NRMSE=0.056)
['GS_TSLA', 'PCR'] for TSLA => (MAE=2.667; RMSE=4.459; ; NRMSE=0.056)
['SC_TSLA', 'VIX'] for TSLA => (MAE=2.683; RMSE=4.450; ; NRMSE=0.056)
['SC_TSLA', 'PCR'] for TSLA => (MAE=2.677; RMSE=4.452; ; NRMSE=0.056)
['VIX', 'PCR'] for TSLA => (MAE=2.699; RMSE=4.495; ; NRMSE=0.057)
['GS_TSLA', 'SC_TSLA', 'VIX'] for TSLA => (MAE=2.604; RMSE=4.371; ; NRMSE=0.055)
['GS_TSLA', 'SC_TSLA', 'PCR'] for TSLA => (MAE=2.595; RMSE=4.393; ; NRMSE=0.055)
['GS_TSLA', 'VIX', 'PCR'] for TSLA => (MAE=2.592; RMSE=4.353; ; NRMSE=0.055)
['SC_TSLA', 'VIX', 'PCR'] for TSLA => (MAE=2.563; RMSE=4.313; ; NRMSE=0.054)
['GS_TSLA', 'SC_TSLA', 'VIX', 'PCR'] for TSLA => (MAE=2.379; RMSE=3.980; ; NRMSE=0.050)
